In [3]:
import pandas as pd

In [5]:
master_df = pd.read_csv("../master_ncaa.csv")

In [161]:
train_years = [13,14,15,16,17,18,19,21,22,23,24]
regions = ['east','south','midwest','west']

In [39]:
df_past64 = []
for year in train_years:
    for region in regions:
        df_past64.append(master_df[(master_df.REGION == region) & (master_df.YEAR ==2000+year)].sort_values('SEED'))
        print(year, region)

13 east
13 south
13 midwest
13 west
14 east
14 south
14 midwest
14 west
15 east
15 south
15 midwest
15 west
16 east
16 south
16 midwest
16 west
17 east
17 south
17 midwest
17 west
18 east
18 south
18 midwest
18 west
19 east
19 south
19 midwest
19 west
21 east
21 south
21 midwest
21 west
22 east
22 south
22 midwest
22 west
23 east
23 south
23 midwest
23 west
24 east
24 south
24 midwest
24 west


In [26]:
train_columns = [
 'G',
 'W',
 "WIN_PER",
 'ADJOE',
 'ADJDE',
 'BARTHAG',
 'EFG_O',
 'EFG_D',
 'TOR',
 'TORD',
 'ORB',
 'DRB',
 'FTR',
 'FTRD',
 '2P_O',
 '2P_D',
 '3P_O',
 '3P_D',
 'ADJ_T',
 'WAB',
 'SEED',
 "POWER"
 ]

In [19]:
def get_upset_differences(a,b):
    """
    Takes two rows from one of the region databases and finds the difference between the two columns 

    a: higher seed
    b: lower seed'

    output: list of db difference rows
    """
    
    listA = []
    for x in range(3,25):
        diff = a.iloc[x] - b.iloc[x]
        listA.append(diff)
    return listA

In [16]:
def get_target_variable(df, nxt_round_num, rnd_name):
    """
    This function used to get the target variable attributes from the correct column
    The list is reversed because we want to see if the lower seed wins

    df: dataframe used
    nxt_round_num: number of teams in the next round
    rnd_name: the column name of the dummy round variable of the next round

    output: list of target variable attributes (will be its own column)
    """
    
    listB = list(df[rnd_name])
    listB.reverse()
    listB = listB[0:nxt_round_num]
    return listB

In [32]:
def create_training_record(df, matchup_num, reseed = True):
    """
    This function takes the dataframe, and creates a new dataset of differences that can be trained
    
    df: dataframe of the round being processed
    matchup_num: the number of matchups
    """
    
    listDF = []
    for y in range(0,matchup_num):
        test_upsetH = df.iloc[y]
        test_upsetL = df.iloc[-(y+1)]
        listDF.append(get_upset_differences(test_upsetH,test_upsetL))
    if reseed:
        return pd.DataFrame(listDF, columns = train_columns).sort_values(by = ["SEED"])
    else:
        return pd.DataFrame(listDF, columns = train_columns).sort_values(by = ["SEED"])

In [33]:
def creation(df, next_rounds):
    """
    This function takes the current round's dataframe and the next round string name to use the previous functions to build the training df

    df: current round's dataframe
    next_rounds
    """
    
    y = pd.DataFrame(columns = train_columns)
    asd = []
    nxt_round_num = int(len(df)/2)
    upset= get_target_variable(df,nxt_round_num, next_rounds)
    for x in range(0,nxt_round_num):
        h = df.iloc[x]
        l = df.iloc[(-x-1)]
        if upset[x] == 1:
            asd.append(l)
        if upset[x] == 0:
            asd.append(h)
    next_df = pd.DataFrame(asd)
    y = pd.concat([y, create_training_record(df,nxt_round_num)], ignore_index=True)
    y["TRAIN"] = upset
    y.TRAIN = y.TRAIN.astype(int)
    return y, next_df

In [34]:
def create_train(a, next_round):
    df = pd.DataFrame(columns = train_columns)
    df_next = []
    for x in a:
        train, next_df = creation(x,next_round)
        df = pd.concat([df,train], ignore_index = True)
        df_next.append(next_df)
    return df, df_next

In [40]:
round_name = ["R32", "S16", "E8", "F4"]
train_master = pd.DataFrame(columns = train_columns)
train_w1 = pd.DataFrame(columns = train_columns)
train_w2 = pd.DataFrame(columns = train_columns)

for x in range(0,len(round_name)):
    a = 2**(6-x)
    b = 2**(5-x)
    holder = create_train(globals()["df_past" + '%s' % a], round_name[x])
    globals()["train" + '%s' % a] = holder[0]
    globals()["df_past" + '%s' % b] = holder[1]
    train_master = pd.concat([train_master,holder[0]], ignore_index = True)
    print("end")
    if x <= 1:
        train_w1 = pd.concat([train_w1,holder[0]], ignore_index = True)
    else:
        train_w2 = pd.concat([train_w2,holder[0]], ignore_index = True)

/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_38649/2317163237.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  y = pd.concat([y, create_training_record(df,nxt_round_num)], ignore_index=True)
/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_38649/3673862151.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,train], ignore_index = True)
/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_38649/2317163237.py:21: FutureWarning: The behavior of D

end
end
end
end


In [116]:
import json

In [117]:
with open("final_four.json", "r") as file:
    f4 = json.load(file)

In [118]:
df_past4 = []
for year in train_years:
        order = f4[str(year+2000)]
        df = master_df[ (master_df.YEAR ==2000+year) & (master_df.games_won >= 5)].sort_values('SEED')
        df = df.set_index('TEAM').reindex(order).reset_index()

        df_past4.append(df)


In [119]:
df_past4

[          TEAM CONF POSTSEASON   G   W   WIN_PER  ADJOE  ADJDE  BARTHAG  \
 0   Louisville   BE  Champions  40  35  0.875000  115.9   84.5   0.9743   
 1     Michigan  B10        2ND  38  30  0.789474  121.5   93.7   0.9522   
 2     Syracuse   BE         F4  40  30  0.750000  113.0   87.0   0.9533   
 3  Wichita St.  MVC         F4  39  30  0.769231  110.6   91.0   0.9045   
 
    EFG_O  ...  YEAR  R64  R32  S16  E8  F4  C2  Champions   REGION  games_won  
 0   50.6  ...  2013    1    1    1   1   1   1          1  midwest          7  
 1   54.6  ...  2013    1    1    1   1   1   1          0    south          6  
 2   49.0  ...  2013    1    1    1   1   1   0          0     east          5  
 3   50.0  ...  2013    1    1    1   1   1   0          0     west          5  
 
 [4 rows x 35 columns],
           TEAM  CONF POSTSEASON   G   W   WIN_PER  ADJOE  ADJDE  BARTHAG  \
 0     Kentucky   SEC        2ND  40  29  0.725000  117.2   96.2   0.9062   
 1      Florida   SEC         F4 

In [120]:
train4, df_past2 = create_train(df_past4, "C2")
train2, df_past1 = create_train(df_past2, "Champions")

/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_38649/2317163237.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  y = pd.concat([y, create_training_record(df,nxt_round_num)], ignore_index=True)
/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_38649/3673862151.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,train], ignore_index = True)
/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_38649/2317163237.py:21: FutureWarning: The behavior of D

In [122]:
train_ff = pd.DataFrame(columns = train_columns)

for x in [train4,train2]:
    train_master = pd.concat([train_master,x], ignore_index = True)
    train_ff = pd.concat([train_ff, x], ignore_index = True)

/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_38649/4269691304.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  train_ff = pd.concat([train_ff, x], ignore_index = True)


In [142]:
def reset_positive_seeds(df):
    train_neg = df[df["SEED"]<=0]
    train_pos = df[df["SEED"]>0]
    train_pos = - train_pos
    train_pos["TRAIN"] = train_pos["TRAIN"] + 1
    return pd.concat([train_pos, train_neg]).reset_index(drop = True)


In [143]:
train_master= reset_positive_seeds(train_master)
train_ff = reset_positive_seeds(train_ff)
train_w1 = reset_positive_seeds(train_w1)
train_w2 = reset_positive_seeds(train_w2)

In [295]:
seed_cutoff_low = -7
seed_cutoff_high = -4
big_upset = train_master[train_master["SEED"] < seed_cutoff_low]
competative = train_master[train_master["SEED"] > seed_cutoff_high]
little_upset = train_master[(train_master["SEED"] <= seed_cutoff_high) & (train_master["SEED"] >= seed_cutoff_low)]

In [296]:
for i, x in [(train_master, 'master'), (train_ff, 'ff'), (train_w1, 'w1'), (train_w2, 'w2'), (big_upset, 'big_upset'), (little_upset, 'little_upset'), (competative, 'competative')]:
    i.to_csv("data/training/" + x + ".csv", index = False)

east25_df

In [6]:
regions = ['east','south','west','midwest']

In [7]:
test_regions = []
for region in regions:
    test_regions.append(master_df[(master_df.REGION == region) & (master_df.YEAR ==2025)].sort_values('SEED'))


In [8]:
df_headers = ['TEAM',
 'CONF',
 'POSTSEASON',
 'G',
 'W',
 'WIN_PER',
 'ADJOE',
 'ADJDE',
 'BARTHAG',
 'EFG_O',
 'EFG_D',
 'TOR',
 'TORD',
 'ORB',
 'DRB',
 'FTR',
 'FTRD',
 '2P_O',
 '2P_D',
 '3P_O',
 '3P_D',
 'ADJ_T',
 'WAB',
 'SEED',
 'POWER',
 'YEAR']

In [9]:
train_columns = df_headers[3:25]

In [10]:
r64_r = pd.DataFrame(columns = df_headers)
r32_r = pd.DataFrame(columns = df_headers)
s16_r = pd.DataFrame(columns = df_headers)
e8_r = pd.DataFrame(columns = df_headers)
f4_r = pd.DataFrame(columns = df_headers)
c2_r= pd.DataFrame(columns = df_headers)
winner_r= pd.DataFrame(columns = df_headers)

In [11]:
test_df = pd.DataFrame(columns = train_columns)
y_pred = []
matchup_list = []

In [12]:
import joblib

In [13]:
random_state = 69

In [15]:
scaler = joblib.load('../models/my_scaler.pkl')  # load from disk


In [16]:
def scale(df):
    m = pd.DataFrame(scaler.transform(df), columns = df.columns)
    return m

In [17]:
w1 = joblib.load('../models/gnb/w1.pkl')  # load from disk
w2 = joblib.load('../models/gnb/w2.pkl')  # load from disk
master = joblib.load('../models/clf/master.pkl')  # load from disk


In [20]:
for x in test_regions: 
    r64_r = pd.concat([r64_r,x], ignore_index = True)
    round64 = x
    rounds = [round64, r32_r, s16_r, e8_r, f4_r]
    for r in range (0,len(rounds)-1):
        wins = []
        y = len(rounds[r])/2
        y = int(y)
        for x in range(0, y):
            h = rounds[r].iloc[x]
            l = rounds[r].iloc[-x-1]
            holder = pd.DataFrame([get_upset_differences(h,l)], columns = df_headers[3:25])
            if holder.iloc[0]["SEED"] > 0: 
                holder = -holder
                static = h
                h = l
                l = static
            scaled_holder = scale(holder)
            ups = master.predict(scaled_holder)
            ups = list(ups)[0]
            print(h['SEED'], h['TEAM'], ' vs. ', l['SEED'], l['TEAM'])
            if ups == 0: 
                wins.append(h)
                print("Winner:", h['SEED'], h['TEAM'])
            if ups == 1:
                wins.append(l)
                print("Winner:", l['SEED'], l['TEAM'])
            test_df = pd.concat([test_df,holder], ignore_index = True)
            matchup = h['TEAM'] + ' vs. ' + l['TEAM']
            matchup_list.append(matchup)
            y_pred.append(ups)
        rounds[r+1] = pd.DataFrame(data = wins, columns = df_headers)
        print("\n")
    print("_" * 40)

    r32_r = pd.concat([r32_r,rounds[1]], ignore_index = True)
    s16_r = pd.concat([s16_r,rounds[2]], ignore_index = True)
    e8_r = pd.concat([e8_r,rounds[3]], ignore_index = True)
    f4_r = pd.concat([f4_r,rounds[4]], ignore_index = True)

1 Duke  vs.  16 Mount St. Mary’s
Winner: 1 Duke
2 Alabama  vs.  15 Robert Morris
Winner: 2 Alabama
3 Wisconsin  vs.  14 Montana
Winner: 3 Wisconsin
4 Arizona  vs.  13 Akron
Winner: 4 Arizona
5 Oregon  vs.  12 Liberty
Winner: 12 Liberty
6 BYU  vs.  11 VCU
Winner: 11 VCU
7 Saint Mary’s  vs.  10 Vanderbilt
Winner: 7 Saint Mary’s
8 Mississippi St.  vs.  9 Baylor
Winner: 9 Baylor


1 Duke  vs.  9 Baylor
Winner: 1 Duke
2 Alabama  vs.  7 Saint Mary’s
Winner: 2 Alabama
3 Wisconsin  vs.  11 VCU
Winner: 3 Wisconsin
4 Arizona  vs.  12 Liberty
Winner: 12 Liberty


1 Duke  vs.  12 Liberty
Winner: 1 Duke
2 Alabama  vs.  3 Wisconsin
Winner: 3 Wisconsin


1 Duke  vs.  3 Wisconsin
Winner: 1 Duke


________________________________________
1 Auburn  vs.  16 Alabama St.
Winner: 1 Auburn
2 Michigan St.  vs.  15 Bryant
Winner: 2 Michigan St.
3 Iowa St.  vs.  14 Lipscomb
Winner: 3 Iowa St.
4 Texas A&M  vs.  13 Yale
Winner: 4 Texas A&M
5 Michigan  vs.  12 UC San Diego
Winner: 12 UC San Diego
6 Mississippi  vs

/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_52127/354623275.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  test_df = pd.concat([test_df,holder], ignore_index = True)
/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_52127/354623275.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  r32_r = pd.concat([r32_r,rounds[1]], ignore_index = True)
/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_52127/354623275.py:37: FutureWarning: The behavior of DataFrame conc

In [34]:
import numpy as np

repeats = int(len(r32_r)/4)
result = np.repeat(regions, repeats).tolist()

In [52]:
from math import log

In [59]:
6 - (len(r32_r)**2)

-1018

In [60]:
test = {}

In [61]:
test.update({"matchup": 1})

In [62]:
test

{'matchup': 1}

In [45]:
from datetime import datetime

In [49]:
int(datetime.now().timestamp())

1772209737

In [ ]:
r32_r['region'] = result

In [ ]:
test_json = .to_dict(orient="records")

In [43]:
r32_r[['SEED', "TEAM", "region"]].to_json("../Sims/2025/test.json", orient="records", indent = 4)


In [367]:
e8_r

,TEAM,CONF,POSTSEASON,G,W,WIN_PER,ADJOE,ADJDE,BARTHAG,EFG_O,...,FTRD,2P_O,2P_D,3P_O,3P_D,ADJ_T,WAB,SEED,POWER,YEAR


## TRAINING

In [299]:
import sklearn.tree as tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import StandardScaler  

In [300]:
scaler = StandardScaler()
scaler.fit(train_master.drop(columns = ["TRAIN"]))

def scale(df):
    m = pd.DataFrame(scaler.transform(df), columns = df.columns)
    return m

In [301]:
def formatStuff(df):
    y = df["TRAIN"]
    x = scale(df.drop(columns = ["TRAIN"]))
    return (x,y)

In [309]:
train_df = formatStuff(train_master)
train_big = formatStuff(big_upset)
train_little = formatStuff(little_upset)
train_comp = formatStuff(competative)
train_final4 = formatStuff(train_ff)
train_first_week = formatStuff(train_w1)
train_second_week = formatStuff(train_w2)

In [310]:
for p in [train_df, train_big, train_little, train_comp, train_final4, train_first_week, train_second_week]:
    print(sum(p[1])/len(p[0]))
    print(len(p[0]), "\n")

0.3189033189033189
693 

0.17543859649122806
285 

0.4052631578947368
190 

0.43119266055045874
218 

0.3333333333333333
33 

0.30303030303030304
528 

0.3787878787878788
132 



In [325]:
# Data 
train_master_x = train_df[0]
train_master_y = list(train_df[1])

train_big_x = train_big[0]
train_big_y = list(train_big[1])

train_little_x = train_little[0]
train_little_y = list(train_little[1])

train_comp_x = train_comp[0]
train_comp_y = list(train_comp[1])

train_ff_x = train_final4[0]
train_ff_y = list(train_final4[1])

train_w1_x = train_first_week[0]
train_w1_y = list(train_first_week[1])

train_w2_x = train_second_week[0]
train_w2_y = list(train_second_week[1])

In [326]:
#MLP Classifier
iterations = 1000
alpha = 3

mlp_master = MLPClassifier(max_iter= iterations, alpha=alpha, random_state = 69)
mlp_master.fit(train_master_x, train_master_y)
print("Accuracy on overall training set: {:.3f}".format(mlp_master.score(train_master_x, train_master_y)))

mlp_big = MLPClassifier(max_iter= iterations, alpha=alpha, random_state = 69)
mlp_big.fit(train_big_x, train_big_y)
print("Accuracy on big upset training set: {:.3f}".format(mlp_big.score(train_big_x, train_big_y)))

mlp_little = MLPClassifier(max_iter= iterations, alpha= alpha, random_state = 69)
mlp_little.fit(train_little_x, train_little_y)
print("Accuracy on little upset training set: {:.3f}".format(mlp_little.score(train_little_x, train_little_y)))

mlp_comp = MLPClassifier(max_iter= iterations, alpha= alpha, random_state = 69)
mlp_comp.fit(train_comp_x, train_comp_y)
print("Accuracy on competative training set: {:.3f}".format(mlp_comp.score(train_comp_x, train_comp_y)))

mlp_ff = MLPClassifier(max_iter= iterations, alpha= alpha, random_state = 69)
mlp_ff.fit(train_ff_x, train_ff_y)
print("Accuracy on final four training set: {:.3f}".format(mlp_ff.score(train_ff_x, train_ff_y)))

mlp_w1 = MLPClassifier(max_iter= iterations, alpha= alpha, random_state = 69)
mlp_w1.fit(train_w1_x, train_w1_y)
print("Accuracy on week 1 training set: {:.3f}".format(mlp_w1.score(train_w1_x, train_w1_y)))

mlp_w2 = MLPClassifier(max_iter= iterations, alpha= alpha, random_state = 69)
mlp_w2.fit(train_w2_x, train_w2_y)
print("Accuracy on week 2 training set: {:.3f}".format(mlp_w2.score(train_w2_x, train_w2_y)))

Accuracy on overall training set: 0.781
Accuracy on big upset training set: 0.863
Accuracy on little upset training set: 0.942
Accuracy on competative training set: 0.775
Accuracy on final four training set: 0.970
Accuracy on week 1 training set: 0.801
Accuracy on week 2 training set: 0.947


In [327]:
est = 6
depth = 5
 
forest_master = RandomForestClassifier(n_estimators=est, max_depth = depth, random_state = 69)
forest_master.fit(train_master_x, train_master_y)
print("Accuracy on training set: {:.3f}".format(
    forest_master.score(train_master_x, train_master_y)))

forest_big = RandomForestClassifier(n_estimators=est, max_depth = depth, random_state = 69)
forest_big.fit(train_big_x, train_big_y)
print("Accuracy on training set: {:.3f}".format(
    forest_big.score(train_big_x, train_big_y)))

forest_little = RandomForestClassifier(n_estimators=est, max_depth = depth, random_state = 69)
forest_little.fit(train_little_x, train_little_y)
print("Accuracy on training set: {:.3f}".format(
    forest_little.score(train_little_x, train_little_y)))

forest_comp = RandomForestClassifier(n_estimators=est, max_depth = depth, random_state = 69)
forest_comp.fit(train_comp_x, train_comp_y)
print("Accuracy on training set: {:.3f}".format(forest_comp.score(train_comp_x, train_comp_y)))


forest_ff = RandomForestClassifier(n_estimators=est, max_depth = depth, random_state = 69)
forest_ff.fit(train_ff_x, train_ff_y)
print("Accuracy on training set: {:.3f}".format(forest_ff.score(train_ff_x, train_ff_y)))


forest_w1 = RandomForestClassifier(n_estimators=est, max_depth = depth, random_state = 69)
forest_w1.fit(train_w1_x, train_w1_y)
print("Accuracy on training set: {:.3f}".format(forest_w1.score(train_w1_x, train_w1_y)))


forest_w2 = RandomForestClassifier(n_estimators=est, max_depth = depth, random_state = 69)
forest_w2.fit(train_w2_x, train_w2_y)
print("Accuracy on training set: {:.3f}".format(forest_w2.score(train_w2_x, train_w2_y)))

Accuracy on training set: 0.830
Accuracy on training set: 0.898
Accuracy on training set: 0.858
Accuracy on training set: 0.885
Accuracy on training set: 0.970
Accuracy on training set: 0.835
Accuracy on training set: 0.909


In [328]:
#SVM

svc_master = SVC(random_state = 69, C = 1)
svc_master.fit(train_master_x, train_master_y)
print("Accuracy on training set: {:.3f}".format(svc_master.score(train_master_x, train_master_y)))

svc_big = SVC(random_state = 69, C = 1)
svc_big.fit(train_big_x, train_big_y)
print("Accuracy on training set: {:.3f}".format(svc_big.score(train_big_x, train_big_y)))

svc_little = SVC(random_state = 69, C = 1)
svc_little.fit(train_little_x, train_little_y)
print("Accuracy on training set: {:.3f}".format(svc_little.score(train_little_x, train_little_y)))

svc_comp = SVC(random_state = 69, C = 1)
svc_comp.fit(train_comp_x, train_comp_y)
print("Accuracy on training set: {:.3f}".format(svc_comp.score(train_comp_x, train_comp_y)))

svc_ff = SVC(random_state = 69, C = 1)
svc_ff.fit(train_ff_x, train_ff_y)
print("Accuracy on training set: {:.3f}".format(svc_ff.score(train_ff_x, train_ff_y)))

svc_w1 = SVC(random_state = 69, C = 1)
svc_w1.fit(train_w1_x, train_w1_y)
print("Accuracy on training set: {:.3f}".format(svc_w1.score(train_w1_x, train_w1_y)))

svc_w2 = SVC(random_state = 69, C = 1)
svc_w2.fit(train_w2_x, train_w2_y)
print("Accuracy on training set: {:.3f}".format(svc_w2.score(train_w2_x, train_w2_y)))

Accuracy on training set: 0.818
Accuracy on training set: 0.828
Accuracy on training set: 0.816
Accuracy on training set: 0.881
Accuracy on training set: 1.000
Accuracy on training set: 0.816
Accuracy on training set: 0.879


In [329]:
#Log Regressor

clf_master =  LogisticRegression(random_state=69, C =1)
clf_master.fit(train_master_x, train_master_y)
print("Accuracy on training set: {:.3f}".format(clf_master.score(train_master_x, train_master_y)))

clf_big =  LogisticRegression(random_state=69, C =1)
clf_big.fit(train_big_x, train_big_y)
print("Accuracy on training set: {:.3f}".format(clf_big.score(train_big_x, train_big_y)))

clf_little = LogisticRegression(random_state=69, C = 5)
clf_little.fit(train_little_x, train_little_y)
print("Accuracy on training set: {:.3f}".format(clf_little.score(train_little_x, train_little_y)))

clf_comp = LogisticRegression(random_state=69, C = 20)
clf_comp.fit(train_comp_x, train_comp_y)
print("Accuracy on training set: {:.3f}".format(clf_comp.score(train_comp_x, train_comp_y)))

clf_ff =  LogisticRegression(random_state=69, C =1)
clf_ff.fit(train_ff_x, train_ff_y)
print("Accuracy on training set: {:.3f}".format(clf_ff.score(train_ff_x, train_ff_y)))

clf_w1 = LogisticRegression(random_state=69, C = 5)
clf_w1.fit(train_w1_x, train_w1_y)
print("Accuracy on training set: {:.3f}".format(clf_w1.score(train_w1_x, train_w1_y)))

clf_w2 = LogisticRegression(random_state=69, C = 20)
clf_w2.fit(train_w2_x, train_w2_y)
print("Accuracy on training set: {:.3f}".format(clf_w2.score(train_w2_x, train_w2_y)))

Accuracy on training set: 0.769
Accuracy on training set: 0.856
Accuracy on training set: 0.742
Accuracy on training set: 0.807
Accuracy on training set: 0.939
Accuracy on training set: 0.759
Accuracy on training set: 0.864


In [330]:
# KNN 
neighbors = 5

knn_master = KNeighborsClassifier(n_neighbors=neighbors)
knn_master.fit(train_master_x, train_master_y)
knn_mscore = knn_master.score(train_master_x, train_master_y)
print("Accuracy on training set: {:.3f}".format(knn_mscore))


knn_big = KNeighborsClassifier(n_neighbors=neighbors)
knn_big.fit(train_big_x, train_big_y)
knn_bscore = knn_big.score(train_big_x, train_big_y)
print("Accuracy on training set: {:.3f}".format(knn_bscore))


knn_little = KNeighborsClassifier(n_neighbors=neighbors)
knn_little.fit(train_little_x, train_little_y)
knn_lscore = knn_little.score(train_little_x, train_little_y)
print("Accuracy on training set: {:.3f}".format(knn_lscore))


knn_comp = KNeighborsClassifier(n_neighbors=neighbors)
knn_comp.fit(train_comp_x, train_comp_y)
knn_cscore = knn_comp.score(train_comp_x, train_comp_y)
print("Accuracy on training set: {:.3f}".format(knn_cscore))


knn_ff = KNeighborsClassifier(n_neighbors=neighbors)
knn_ff.fit(train_ff_x, train_ff_y)
knn_fscore = knn_ff.score(train_ff_x, train_ff_y)
print("Accuracy on training set: {:.3f}".format(knn_fscore))

knn_w1 = KNeighborsClassifier(n_neighbors=neighbors)
knn_w1.fit(train_w1_x, train_w1_y)
knn_w1_score = knn_w1.score(train_w1_x, train_w1_y)
print("Accuracy on training set: {:.3f}".format(knn_w1_score))

knn_w2 = KNeighborsClassifier(n_neighbors=neighbors)
knn_w2.fit(train_w2_x, train_w2_y)
knn_w2_score = knn_w2.score(train_w2_x, train_w2_y)
print("Accuracy on training set: {:.3f}".format(knn_w2_score))

Accuracy on training set: 0.795
Accuracy on training set: 0.849
Accuracy on training set: 0.758
Accuracy on training set: 0.784
Accuracy on training set: 0.879
Accuracy on training set: 0.786
Accuracy on training set: 0.727


In [331]:
# Decision Tree


tree_depth = 7

DT_master = DecisionTreeClassifier(min_samples_leaf = 5, max_depth = tree_depth)
DT_master.fit(train_master_x, train_master_y)
print("Accuracy on training set: {:.3f}".format(DT_master.score(train_master_x, train_master_y)))

DT_big = DecisionTreeClassifier(min_samples_leaf = 5, max_depth = tree_depth)
DT_big.fit(train_big_x, train_big_y)
print("Accuracy on training set: {:.3f}".format(DT_big.score(train_big_x, train_big_y)))

DT_little = DecisionTreeClassifier(min_samples_leaf = 5, max_depth = tree_depth)
DT_little.fit(train_little_x, train_little_y)
print("Accuracy on training set: {:.3f}".format(DT_little.score(train_little_x, train_little_y)))

DT_comp = DecisionTreeClassifier(min_samples_leaf = 5, max_depth = tree_depth)
DT_comp.fit(train_comp_x, train_comp_y)
print("Accuracy on training set: {:.3f}".format(DT_comp.score(train_comp_x, train_comp_y)))

DT_ff = DecisionTreeClassifier(min_samples_leaf = 5, max_depth = tree_depth)
DT_ff.fit(train_ff_x, train_ff_y)
print("Accuracy on training set: {:.3f}".format(DT_ff.score(train_ff_x, train_ff_y)))

DT_w1 = DecisionTreeClassifier(min_samples_leaf = 5, max_depth = tree_depth)
DT_w1.fit(train_w1_x, train_w1_y)
print("Accuracy on training set: {:.3f}".format(DT_w1.score(train_w1_x, train_w1_y)))

DT_w2 = DecisionTreeClassifier(min_samples_leaf = 5, max_depth = tree_depth)
DT_w2.fit(train_w2_x, train_w2_y)
print("Accuracy on training set: {:.3f}".format(DT_w2.score(train_w2_x, train_w2_y)))


Accuracy on training set: 0.861
Accuracy on training set: 0.905
Accuracy on training set: 0.863
Accuracy on training set: 0.849
Accuracy on training set: 0.909
Accuracy on training set: 0.845
Accuracy on training set: 0.864


In [332]:
# Naive Bayes


gnb_master = GaussianNB()
gnb_master.fit(train_master_x, train_master_y)
print("Accuracy on training set: {:.3f}".format(gnb_master.score(train_master_x, train_master_y)))


gnb_big = GaussianNB()
gnb_big.fit(train_big_x, train_big_y)
print("Accuracy on training set: {:.3f}".format(gnb_big.score(train_big_x, train_big_y)))

gnb_little = GaussianNB()
gnb_little.fit(train_little_x, train_little_y)
print("Accuracy on training set: {:.3f}".format(gnb_little.score(train_little_x, train_little_y)))

gnb_comp = GaussianNB()
gnb_comp.fit(train_comp_x, train_comp_y)
print("Accuracy on training set: {:.3f}".format(gnb_comp.score(train_comp_x, train_comp_y)))

gnb_ff = GaussianNB()
gnb_ff.fit(train_ff_x, train_ff_y)
print("Accuracy on training set: {:.3f}".format(gnb_ff.score(train_ff_x, train_ff_y)))

gnb_w1 = GaussianNB()
gnb_w1.fit(train_w1_x, train_w1_y)
print("Accuracy on training set: {:.3f}".format(gnb_w1.score(train_w1_x, train_w1_y)))

gnb_w2 = GaussianNB()
gnb_w2.fit(train_w2_x, train_w2_y)
print("Accuracy on training set: {:.3f}".format(gnb_w2.score(train_w2_x, train_w2_y)))

Accuracy on training set: 0.670
Accuracy on training set: 0.744
Accuracy on training set: 0.642
Accuracy on training set: 0.711
Accuracy on training set: 0.879
Accuracy on training set: 0.667
Accuracy on training set: 0.750


In [341]:
acc = []

In [346]:
import pickle

In [349]:
for model_type in ['knn', 'DT', 'forest', 'mlp', 'clf', 'gnb', 'svc']:
    for training_set in ['master', 'ff', 'w1', 'w2','big', 'little','comp']:
        X_train = globals()["train_" + training_set + "_x"]
        y_train = globals()["train_" + training_set + "_y"]
        model = globals()[model_type + "_" + training_set]

        model.fit(X_train, y_train)
        print("Accuracy on training set: {:.3f}".format(model.score(X_train, y_train)) + " for " + str(model) + " on " + training_set)
        acc.append({"model_type": model_type, "training_set": training_set, "accuracy": model.score(X_train, y_train)})

        filename = f'models/{model_type}/{training_set}_model.sav'
        pickle.dump(model, open(filename, 'wb'))


Accuracy on training set: 0.795 for KNeighborsClassifier() on master
Accuracy on training set: 0.879 for KNeighborsClassifier() on ff
Accuracy on training set: 0.786 for KNeighborsClassifier() on w1
Accuracy on training set: 0.727 for KNeighborsClassifier() on w2
Accuracy on training set: 0.849 for KNeighborsClassifier() on big
Accuracy on training set: 0.758 for KNeighborsClassifier() on little
Accuracy on training set: 0.784 for KNeighborsClassifier() on comp
Accuracy on training set: 0.859 for DecisionTreeClassifier(max_depth=7, min_samples_leaf=5) on master
Accuracy on training set: 0.909 for DecisionTreeClassifier(max_depth=7, min_samples_leaf=5) on ff
Accuracy on training set: 0.845 for DecisionTreeClassifier(max_depth=7, min_samples_leaf=5) on w1
Accuracy on training set: 0.864 for DecisionTreeClassifier(max_depth=7, min_samples_leaf=5) on w2
Accuracy on training set: 0.905 for DecisionTreeClassifier(max_depth=7, min_samples_leaf=5) on big
Accuracy on training set: 0.863 for Dec

In [345]:
pd.DataFrame(acc).sort_values(by = ["accuracy"], ascending = False)

,model_type,training_set,accuracy
43,svc,ff,1.000000
22,mlp,ff,0.969697
15,forest,ff,0.969697
24,mlp,w2,0.946970
26,mlp,little,0.942105
29,clf,ff,0.939394
8,DT,ff,0.909091
17,forest,w2,0.909091
11,DT,big,0.905263
18,forest,big,0.898246
